# 🏊 YOLOv7 Pose Detection — Swimmer Digital Twin POC

This notebook runs YOLOv7 pose detection on a swimming video and extracts
COCO17-keypoint skeleton data per frame.

**COCO Keypoints:**
0: nose, 1: left_eye, 2: right_eye, 3: left_ear, 4: right_ear,
5: left_shoulder, 6: right_shoulder, 7: left_elbow, 8: right_elbow,
9: left_wrist, 10: right_wrist, 11: left_hip, 12: right_hip,
13: left_knee, 14: right_knee, 15: left_ankle, 16: right_ankle

---

## 1. Setup — Clone YOLOv7 & download weights

In [ ]:
!git clone https://github.com/WongKinYiu/yolov7.git
%cd yolov7
!wget -q https://github.com/WongKinYiu/yolov7/releases/download/v0.1/yolov7-w6-pose.pt -O yolov7-w6-pose.pt
print('Done — model ready')

## 2. Upload swimming video

In [ ]:
from google.colab import files
uploaded = files.upload()
video_path = list(uploaded.keys())[0]
print(f'Video uploaded: {video_path}')

## 3. Extract frames from video

In [ ]:
import cv2
import os
from pathlib import Path

frames_dir = Path('/content/frames')
frames_dir.mkdir(exist_ok=True)

vid = cv2.VideoCapture(video_path)
fps = vid.get(cv2.CAP_PROP_FPS)
total_frames = int(vid.get(cv2.CAP_PROP_FRAME_COUNT))
print(f'FPS: {fps:.1f}  |  Total frames: {total_frames}')

frame_paths = []
frame_idx = 0
while True:
    ret, frame = vid.read()
    if not ret:
        break
    # Save every N frames to keep it manageable (e.g.,1 per second)
    if frame_idx % max(1, int(fps)) == 0:
        path = frames_dir / f'frame_{frame_idx:06d}.jpg'
        cv2.imwrite(str(path), frame)
        frame_paths.append(path)
    frame_idx += 1

vid.release()
print(f'Extracted {len(frame_paths)} frames → {frames_dir}')

## 4. Run YOLOv7 pose on every frame

In [ ]:
import torch
import numpy as np
from torchvision import transforms

# Load model
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

model = torch.load('yolov7-w6-pose.pt', map_location=device)
model.eval()

def run_pose(img_path):
    """Run pose detection on a single image. Returns list of keypoint arrays."""
    img = cv2.imread(str(img_path))
    orig_h, orig_w = img.shape[:2]

    # Preprocess
    img_resized = cv2.resize(img, (640, 640))
    img_rgb = cv2.cvtColor(img_resized, cv2.COLOR_BGR2RGB)
    img_tensor = torch.from_numpy(img_rgb).float() / 255.0
    img_tensor = img_tensor.permute(2, 0, 1).unsqueeze(0).to(device)

    with torch.no_grad():
 outputs = model(img_tensor)

    # Parse pose output (COCO 17 keypoints)
    # output shape: (batch, 17*3, num_proposals) — keypoints (x, y, conf)
    keypoints_list = []
    for kp in outputs[0]:
        # kp: (51,) → reshape to (17, 3)
        kp_arr = kp.cpu().numpy().reshape(17, 3)
        # Scale back to original image size
        kp_arr[:, 0] *= orig_w / 640
        kp_arr[:, 1] *= orig_h / 640
        keypoints_list.append(kp_arr)

    return keypoints_list

print('Model loaded — ready for inference')

## 5. Run inference on all frames

In [ ]:
import json
from tqdm import tqdm

results = []
for fp in tqdm(frame_paths, desc='Processing frames'):
    keypoints = run_pose(fp)
    frame_data = {
        'frame_file': fp.name,
        'frame_idx': int(fp.stem.split('_')[1]),
        'num_detections': len(keypoints),
        'keypoints': [kp.tolist() for kp in keypoints]
    }
    results.append(frame_data)

print(f'\nProcessed {len(results)} frames')

## 6. Save results to JSON

In [ ]:
output_path = '/content/pose_results.json'
with open(output_path, 'w') as f:
    json.dump(results, f, indent=1)

print(f'Results saved → {output_path}')

# Download
files.download(output_path)
print('Download triggered — save pose_results.json to the project data/ folder')

## 7. Quick visual check — overlay skeleton on a few frames

In [ ]:
import matplotlib.pyplot as plt

KPNAMES = [
    'nose','left_eye','right_eye','left_ear','right_ear',
    'left_shoulder','right_shoulder','left_elbow','right_elbow',
    'left_wrist','right_wrist','left_hip','right_hip',
    'left_knee','right_knee','left_ankle','right_ankle'
]

# Skeleton connections (COCO)
SKELETON = [
    (5,6),(5,7),(7,9),(6,8),(8,10),
    (5,11),(6,12),(11,12),
    (11,13),(13,15),(12,14),(14,16)
]

def draw_skeleton(img, keypoints, conf_thresh=0.3):
    for kp in keypoints:
        if kp[0,2] < conf_thresh:
            continue
        for a, b in SKELETON:
            if kp[a,2] > conf_thresh and kp[b,2] > conf_thresh:
                cv2.line(img, (int(kp[a,0]), int(kp[a,1])), (int(kp[b,0]), int(kp[b,1])), (0,255,0), 2)
        for i, pt in enumerate(kp):
            if pt[2] > conf_thresh:
                cv2.circle(img, (int(pt[0]), int(pt[1])), 4, (0,0,255), -1)
                cv2.putText(img, KPNAMES[i], (int(pt[0])+5, int(pt[1])), cv2.FONT_HERSHEY_SIMPLEX, 0.4, (255,255,0), 1)
    return img

# Show first 4 frames with detections
fig, axes = plt.subplots(1, 4, figsize=(20, 5))
for ax, fp in zip(axes, frame_paths[:4]):
    kps = run_pose(fp)
    img = cv2.imread(str(fp))
    img_skel = draw_skeleton(img, kps)
    img_rgb = cv2.cvtColor(img_skel, cv2.COLOR_BGR2RGB)
    ax.imshow(img_rgb)
    ax.set_title(fp.name)
    ax.axis('off')
plt.tight_layout()
plt.show()

## 8. Download annotated frames + full video

To download individual frames or the full annotated video,
uncomment and run below:

```python
# Download a zip of all frames
!zip -r /content/frames.zip /content/frames
files.download('/content/frames.zip')
```